In [14]:
import os
import json
from datetime import date
from typing import TypedDict, Annotated
from dotenv import load_dotenv

load_dotenv()


True

In [4]:
!pip install langchain-huggingface -q

In [35]:
# ── LLM: HuggingFace instead of OpenAI ───────────────────────────────────────
# This is the ONLY change from the original — swap ChatOpenAI for this
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

HF_TOKEN = os.environ.get("HUGGINGFACE_API_TOKEN")
print("Token loaded:", HF_TOKEN[:10], "...")  # verify it loaded

llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    huggingfacehub_api_token=HF_TOKEN,
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1,
)

llm = ChatHuggingFace(llm=llm_endpoint)

Token loaded: hf_VVYpOcI ...


In [36]:
# ── LangGraph & LangChain imports ─────────────────────────────────────────────
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages


In [10]:
!pip install tavily-python -q


In [37]:
# Tavily (web search)
from tavily import TavilyClient

In [38]:
# ── State (same pattern as original) ─────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]   # LangGraph manages message history

# ── Tool 1: Current date (same as original) ───────────────────────────────────
@tool
def get_current_date_tool() -> str:
    """Returns the current date in 'YYYY-MM-DD' format.
    Useful for finding flights/hotels relative to today."""
    return date.today().isoformat()

# ── Tool 2: Addition & Subtraction (practice task from original) ──────────────
@tool
def math_tool(expression: str) -> str:
    """Performs addition or subtraction.
    Pass the expression as a string, e.g. '5 + 3' or '10 - 4'."""
    import re
    add = re.search(r"(-?\d+)\s*\+\s*(-?\d+)", expression)
    sub = re.search(r"(-?\d+)\s*-\s*(-?\d+)", expression)
    if add:
        a, b = int(add.group(1)), int(add.group(2))
        return f"{a} + {b} = {a + b}"
    elif sub:
        a, b = int(sub.group(1)), int(sub.group(2))
        return f"{a} - {b} = {a - b}"
    return "Could not parse expression. Use format like '5 + 3' or '10 - 4'."

# ── Tool 3: Tavily web search ─────────────────────────────────────────────────
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

@tool
def tavily_search_tool(query: str) -> str:
    """Searches the web for current news, travel advisories, and general information."""
    response = tavily_client.search(query=query, max_results=3)
    results = response.get("results", [])
    if not results:
        return "No results found."
    return "\n\n".join(r["content"][:400] for r in results)

In [39]:
# ── Bind tools to LLM — EXACT same pattern as original ───────────────────────
tools_list = [
    get_current_date_tool,
    math_tool,
    tavily_search_tool
]

In [40]:
llm_with_tools = llm.bind_tools(tools_list)  # ← same line as original, just llm changed


In [41]:
# ── Graph nodes — same as original's build_graph_one_tool() ──────────────────
def call_llm(state: AgentState) -> AgentState:
    """LLM decides: call a tool OR produce final answer."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    """
    Routing function — same logic as original.
    If LLM called a tool → go to tools node.
    If LLM is done → END.
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# ToolNode automatically executes whichever tool the LLM chose
tool_node = ToolNode(tools_list)

In [42]:
# ── Build the graph — same structure as original's build_graph_one_tool() ─────
def build_graph_one_tool(tools):
    """Mirrors the original build_graph_one_tool() function exactly."""
    # Re-bind with whatever tools are passed
    _llm_with_tools = llm.bind_tools(tools)
    _tool_node = ToolNode(tools)

    def _call_llm(state: AgentState):
        response = _llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    workflow = StateGraph(AgentState)
    workflow.add_node("llm", _call_llm)
    workflow.add_node("tools", _tool_node)

    workflow.set_entry_point("llm")
    workflow.add_conditional_edges(
        "llm",
        lambda s: "tools" if (
            hasattr(s["messages"][-1], "tool_calls") and s["messages"][-1].tool_calls
        ) else END,
    )
    workflow.add_edge("tools", "llm")   # after tool runs → back to LLM to decide next step

    return workflow.compile()

In [43]:
# ── app_call helper — same as original ───────────────────────────────────────
def app_call(app, prompt: str):
    """Invokes the graph and returns (final_output, full_history)."""
    initial_state = {"messages": [HumanMessage(content=prompt)]}
    result = app.invoke(initial_state)
    history = result["messages"]
    final_output = history[-1].content
    return final_output, history

In [45]:
# ── Run — mirrors original notebook cells exactly ─────────────────────────────
if __name__ == "__main__":

        # ── Cell 1: Date tool only
    print("=" * 60)
    print("TEST 1: Date tool")
    app_current_date = build_graph_one_tool([get_current_date_tool])
    output, history = app_call(app_current_date, "What is the current date?")
    print("Output:", output)

    # ── Cell 2: Math tool (practice task from original)
    print("\n" + "=" * 60)
    print("TEST 2: Math tool (practice task)")
    app_math = build_graph_one_tool([get_current_date_tool, math_tool])
    output, history = app_call(app_math, "What is 47 + 83?")
    print("Output:", output)
    for msg in history:
        print(f"[{msg.__class__.__name__}]: {str(msg.content)[:120]}")

    # ── Cell 3: Full travel agent (Tavily only, no Amadeus)
    print("\n" + "=" * 60)
    print("TEST 3: Full travel agent")
    tools_all = [tavily_search_tool, get_current_date_tool]
    app_travel_agent = build_graph_one_tool(tools_all)
    output, history = app_call(
        app_travel_agent,
        "I want the latest news about Cairo, I'm planning to visit from 2026-06-01 to 2026-06-04. "
        "Please fetch security and travel advisories. Finally, format the combined output."
    )
    print("\n==================== OUTPUT ====================")
    print(output)
    print("\n==================== HISTORY ===================")
    for msg in history:
        print(f"[{msg.__class__.__name__}]: {str(msg.content)[:120]}")

TEST 1: Date tool
Output: The current date is 2026-03-26.

TEST 2: Math tool (practice task)
Output: The sum of 47 and 83 is 130.

TEST 3: Full travel agent

==================== OUTPUT ====================
### Latest News About Cairo:
- **Political Developments:** CIA Director William Burns is in Cairo attempting to resolve a month-long impasse.
- **Cultural Updates:** Egypt has received 36 stolen historic artifacts from the U.S. authorities. However, there is an ongoing investigation into the theft of a second ancient artifact within a month.
- **Regional Impact:** Libya has towed a drifting Russian tanker to prevent an environmental disaster near the coast.

### Travel and Security Advisories for Cairo:
- **Travel Advisory Level:** The U.S. State Department rates Egypt at Level 2 - "Exercise Increased Caution."
- **Specific Areas to Avoid:** The Northern and Middle Sinai, as well as parts of the Western Desert, are under a "Do Not Travel" advisory due to security concerns.
- **Gener

In [48]:
print("\n" + "=" * 60)
print("TEST 2: Math tool (practice task)")
app_math = build_graph_one_tool([get_current_date_tool])
output, history = app_call(app_math, "What is 47 + 83?")
print("Output:", output)
for msg in history:
    print(f"[{msg.__class__.__name__}]: {str(msg.content)[:120]}")



TEST 2: Math tool (practice task)
Output: The sum of 47 and 83 is 130.
[HumanMessage]: What is 47 + 83?
[AIMessage]: The sum of 47 and 83 is 130.
